# 02 Preprocessing and Feature Engineering

This notebook documents the preprocessing and feature engineering stage.

It uses the cleaned AquaSensor and weather datasets to create the final modelling dataset used for dissolved oxygen prediction.

Main steps:
- Load cleaned AquaSensor data
- Load cleaned weather data
- Merge weather with sensor readings
- Create time-based features
- Create season proxy
- Create pollution alert
- Create anomaly type
- Create DO prediction targets from 15 minutes to 120 minutes
- Save the final modelling dataset

Cell 1 - Markdown
Cell 2-12 - Code
Cell 13 - Summary

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer

AQUASENSOR_CLEANED = "../data/processed/aquasensor_cleaned.csv"
WEATHER_CLEANED = "../data/processed/weather_cleaned.csv"
GOVT_CLEANED = "../data/processed/govt_cleaned.csv"

OUTPUT_FILE = "../data/processed/aquasensor_final.csv"

HORIZONS = {
    "15min": 1,
    "30min": 2,
    "45min": 3,
    "60min": 4,
    "75min": 5,
    "90min": 6,
    "105min": 7,
    "120min": 8,
}

In [2]:
df = pd.read_csv(AQUASENSOR_CLEANED, parse_dates=["timestamp"], low_memory=False)

df["sensor_id"] = df["sensor_id"].astype(str)
df["sensor_name"] = df["sensor_name"].astype(str)

df = df.sort_values(["sensor_id", "timestamp"]).reset_index(drop=True)

print("AquaSensor cleaned data loaded")
print("Rows:", len(df))
print("Start:", df["timestamp"].min())
print("End:", df["timestamp"].max())

df.head()

AquaSensor cleaned data loaded
Rows: 107974
Start: 2023-10-30 00:00:00
End: 2026-06-04 21:13:40


,timestamp,sensor_id,sensor_name,temperature,dissolved_oxygen_mgl,dissolved_oxygen_pct
0,2025-06-01 00:04:48,941115,Derwent 13-50,13.0,9.7,91.2
1,2025-06-01 00:16:44,941115,Derwent 13-50,12.9,9.7,91.1
2,2025-06-01 00:28:41,941115,Derwent 13-50,12.9,9.7,90.9
3,2025-06-01 00:40:37,941115,Derwent 13-50,12.8,9.7,90.7
4,2025-06-01 00:52:33,941115,Derwent 13-50,12.8,9.7,90.5


In [3]:
weather = pd.read_csv(WEATHER_CLEANED, parse_dates=["timestamp"], low_memory=False)

df["ts_hr"] = df["timestamp"].dt.floor("h")
weather["ts_hr"] = weather["timestamp"].dt.floor("h")

df = df.merge(
    weather[["ts_hr", "air_temperature_c", "sunshine_wm2", "cloud_cover_pct"]],
    on="ts_hr",
    how="left",
)

df = df.drop(columns=["ts_hr"])

print("Weather data merged")
df[["timestamp", "sensor_name", "temperature", "air_temperature_c", "sunshine_wm2", "cloud_cover_pct"]].head()

Weather data merged


,timestamp,sensor_name,temperature,air_temperature_c,sunshine_wm2,cloud_cover_pct
0,2025-06-01 00:04:48,Derwent 13-50,13.0,11.3,0.0,17.0
1,2025-06-01 00:16:44,Derwent 13-50,12.9,11.3,0.0,17.0
2,2025-06-01 00:28:41,Derwent 13-50,12.9,11.3,0.0,17.0
3,2025-06-01 00:40:37,Derwent 13-50,12.8,11.3,0.0,17.0
4,2025-06-01 00:52:33,Derwent 13-50,12.8,11.3,0.0,17.0


In [4]:
if os.path.exists(GOVT_CLEANED):
    govt = pd.read_csv(GOVT_CLEANED, parse_dates=["timestamp"], low_memory=False)
    print("Government data loaded for context/comparison")
    print("Rows:", len(govt))
    print("Start:", govt["timestamp"].min())
    print("End:", govt["timestamp"].max())
else:
    print("Government cleaned file not found")

Government data loaded for context/comparison
Rows: 8312
Start: 2026-04-21 12:01:11
End: 2026-06-03 20:01:11


In [5]:
df["hour"] = df["timestamp"].dt.hour
df["month"] = df["timestamp"].dt.month
df["day_of_year"] = df["timestamp"].dt.dayofyear

df["season"] = df["month"].map(
    {
        12: 0,
        1: 0,
        2: 0,
        3: 1,
        4: 1,
        5: 1,
        6: 2,
        7: 2,
        8: 2,
        9: 3,
        10: 3,
        11: 3,
    }
)

print("Time features added")
df[["timestamp", "hour", "month", "day_of_year", "season"]].head()

Time features added


,timestamp,hour,month,day_of_year,season
0,2025-06-01 00:04:48,0,6,152,2
1,2025-06-01 00:16:44,0,6,152,2
2,2025-06-01 00:28:41,0,6,152,2
3,2025-06-01 00:40:37,0,6,152,2
4,2025-06-01 00:52:33,0,6,152,2


In [6]:
df["cloud_cover_pct"] = df["cloud_cover_pct"].fillna(50)

angle = 2 * np.pi * (df["day_of_year"] / 365.25)
seasonal_signal = (np.cos(angle - np.pi) + 1) / 2
clear_sky_signal = 1 - (df["cloud_cover_pct"] / 100)

df["season_proxy"] = (
    0.70 * seasonal_signal + 0.30 * clear_sky_signal
).round(4)

print("Season proxy created")
df[["timestamp", "cloud_cover_pct", "season_proxy"]].head()

Season proxy created


,timestamp,cloud_cover_pct,season_proxy
0,2025-06-01 00:04:48,17.0,0.9015
1,2025-06-01 00:16:44,17.0,0.9015
2,2025-06-01 00:28:41,17.0,0.9015
3,2025-06-01 00:40:37,17.0,0.9015
4,2025-06-01 00:52:33,17.0,0.9015


In [7]:
df["pollution_alert"] = (df["dissolved_oxygen_mgl"] < 4.0).astype(int)

print("Pollution alerts created:", df["pollution_alert"].sum())
df[["timestamp", "sensor_name", "dissolved_oxygen_mgl", "pollution_alert"]].head()

Pollution alerts created: 231


,timestamp,sensor_name,dissolved_oxygen_mgl,pollution_alert
0,2025-06-01 00:04:48,Derwent 13-50,9.7,0
1,2025-06-01 00:16:44,Derwent 13-50,9.7,0
2,2025-06-01 00:28:41,Derwent 13-50,9.7,0
3,2025-06-01 00:40:37,Derwent 13-50,9.7,0
4,2025-06-01 00:52:33,Derwent 13-50,9.7,0


In [8]:
df = df.sort_values(["sensor_id", "timestamp"]).reset_index(drop=True)

df["anomaly_type"] = 0

df["gap_min"] = (
    df.groupby("sensor_id")["timestamp"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

df.loc[df["gap_min"] > 24, "anomaly_type"] = 2
df = df.drop(columns=["gap_min"])

df["time_30min"] = df["timestamp"].dt.floor("30min")
low_do = df[df["dissolved_oxygen_mgl"] < 4.0]

if len(low_do) > 0:
    affected = (
        low_do.groupby("time_30min")["sensor_id"]
        .nunique()
        .reset_index()
    )

    affected.columns = ["time_30min", "sensors_affected"]

    df = df.merge(affected, on="time_30min", how="left")
    df["sensors_affected"] = df["sensors_affected"].fillna(0)

    low_mask = df["dissolved_oxygen_mgl"] < 4.0

    df.loc[low_mask & (df["sensors_affected"] == 1), "anomaly_type"] = 1
    df.loc[low_mask & (df["sensors_affected"] >= 2), "anomaly_type"] = 3

    df = df.drop(columns=["sensors_affected"])

df = df.drop(columns=["time_30min"])

print("Anomaly type created")
print(df["anomaly_type"].value_counts().sort_index())

Anomaly type created
anomaly_type
0    107666
1       231
2        77
Name: count, dtype: int64


In [9]:
for horizon, step in HORIZONS.items():
    df[f"do_mgl_next_{horizon}"] = (
        df.groupby("sensor_id")["dissolved_oxygen_mgl"].shift(-step)
    )

    df[f"do_pct_next_{horizon}"] = (
        df.groupby("sensor_id")["dissolved_oxygen_pct"].shift(-step)
    )

print("Prediction targets created")

for horizon in HORIZONS:
    print(
        f"{horizon}: "
        f"mg/L usable = {df[f'do_mgl_next_{horizon}'].notna().sum()}, "
        f"% usable = {df[f'do_pct_next_{horizon}'].notna().sum()}"
    )

Prediction targets created
15min: mg/L usable = 107971, % usable = 107971
30min: mg/L usable = 107968, % usable = 107968
45min: mg/L usable = 107965, % usable = 107965
60min: mg/L usable = 107962, % usable = 107962
75min: mg/L usable = 107959, % usable = 107959
90min: mg/L usable = 107956, % usable = 107956
105min: mg/L usable = 107953, % usable = 107953
120min: mg/L usable = 107950, % usable = 107950


In [10]:
feature_cols = [
    "temperature",
    "air_temperature_c",
    "sunshine_wm2",
    "hour",
    "season_proxy",
]

imputer = KNNImputer(n_neighbors=5)
df[feature_cols] = imputer.fit_transform(df[feature_cols])

print("Missing feature values imputed")
print(df[feature_cols].isnull().sum())

Missing feature values imputed
temperature          0
air_temperature_c    0
sunshine_wm2         0
hour                 0
season_proxy         0
dtype: int64


In [11]:
os.makedirs("../data/processed", exist_ok=True)

df.to_csv(OUTPUT_FILE, index=False)

print("Final modelling dataset saved:")
print(OUTPUT_FILE)

print("\nFinal dataset summary:")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Start:", df["timestamp"].min())
print("End:", df["timestamp"].max())

df.head()

Final modelling dataset saved:
../data/processed/aquasensor_final.csv

Final dataset summary:
Rows: 107974
Columns: 32
Start: 2023-10-30 00:00:00
End: 2026-06-04 21:13:40


,timestamp,sensor_id,sensor_name,temperature,dissolved_oxygen_mgl,dissolved_oxygen_pct,air_temperature_c,sunshine_wm2,cloud_cover_pct,hour,...,do_mgl_next_60min,do_pct_next_60min,do_mgl_next_75min,do_pct_next_75min,do_mgl_next_90min,do_pct_next_90min,do_mgl_next_105min,do_pct_next_105min,do_mgl_next_120min,do_pct_next_120min
0,2025-06-01 00:04:48,941115,Derwent 13-50,13.0,9.7,91.2,11.3,0.0,17.0,0.0,...,9.7,90.5,9.6,90.4,9.6,90.2,9.6,90.1,9.6,89.9
1,2025-06-01 00:16:44,941115,Derwent 13-50,12.9,9.7,91.1,11.3,0.0,17.0,0.0,...,9.6,90.4,9.6,90.2,9.6,90.1,9.6,89.9,9.6,89.8
2,2025-06-01 00:28:41,941115,Derwent 13-50,12.9,9.7,90.9,11.3,0.0,17.0,0.0,...,9.6,90.2,9.6,90.1,9.6,89.9,9.6,89.8,9.6,89.7
3,2025-06-01 00:40:37,941115,Derwent 13-50,12.8,9.7,90.7,11.3,0.0,17.0,0.0,...,9.6,90.1,9.6,89.9,9.6,89.8,9.6,89.7,9.6,89.6
4,2025-06-01 00:52:33,941115,Derwent 13-50,12.8,9.7,90.5,11.3,0.0,17.0,0.0,...,9.6,89.9,9.6,89.8,9.6,89.7,9.6,89.6,9.6,89.5


## Summary

The preprocessing stage produced the final modelling dataset:

`data/processed/aquasensor_final.csv`

This dataset includes cleaned AquaSensor readings, merged weather variables, engineered time and seasonal features, pollution/anomaly labels, and multi-horizon dissolved oxygen targets for both mg/L and percentage.